
---

# 📘 Notebook: ML Project Teaching Guide – Feature Engineering & Ensemble Modeling

---

## 🎓 Objective:
Teach students how to build an end-to-end machine learning pipeline using feature engineering, cross-validation, and ensemble blending — as seen in the Kaggle-style notebook.

---

## 🧰 Prerequisites (Set Prior)

| Topic | Description |
|-------|-------------|
| Python Basics | Variables, loops, functions |
| Pandas | DataFrames, filtering, handling columns |
| Numpy | Array operations, math functions |
| Scikit-learn | Basic understanding of train/test split, metrics |
| Jupyter Notebooks | Comfortable writing and running cells |

### 🔧 Setup Instructions:
```bash
pip install pandas numpy scikit-learn matplotlib seaborn xgboost lightgbm catboost
```

If using GPU:
```bash
pip install --extra-index-url https://pypi.nvidia.com/cudf cudf cuml cuxgboost
```

---

## 🧪 Section 1: Load Data

### Code:
```python
import pandas as pd
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
submission = pd.read_csv("sample_submission.csv")

numerical_features = ['Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp']
```

### 📝 Notes:
- `train.csv`: Contains features + target (`Calories`)
- `test.csv`: Only features; no labels
- `sample_submission.csv`: Template for submission
- `numerical_features`: These will be used for feature engineering

### 💡 Tip:
Use `df.head()` and `df.info()` to explore data after loading.

---

## 🧬 Section 2: Feature Engineering

### Functions Used:
```python
def add_feature_cross_terms(df, features): ...
def add_interaction_features(df, features): ...
def add_statistical_features(df, features): ...
```

### Apply:
```python
train = add_feature_cross_terms(train, numerical_features)
train = add_interaction_features(train, numerical_features)
train = add_statistical_features(train, numerical_features)
```

### 📝 Notes:
| Function | What it does | Why useful |
|---------|--------------|------------|
| `add_feature_cross_terms` | Multiplies pairs of features | Captures multiplicative relationships |
| `add_interaction_features` | Adds/subtracts/divides feature pairs | Finds hidden patterns |
| `add_statistical_features` | Row-wise mean, std, min/max/median | Helps models understand group behavior |

### 📌 Concept:
Feature engineering helps models learn better without changing the model itself.

---

## 🔤 Section 3: Categorical Encoding

```python
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train['Sex'] = le.fit_transform(train['Sex'])
test['Sex'] = le.transform(test['Sex'])
```

### 📝 Notes:
- `'Male'/'Female'` → `0/1`
- Always use `.fit_transform()` on train and `.transform()` on test
- Optional: Use `astype('category')` for memory efficiency

---

## 📐 Section 4: Polynomial Features

```python
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_train = poly.fit_transform(train[numerical_features])
```

### 📝 Notes:
- Generates degree-2 combinations like `Age * Duration`, etc.
- `interaction_only=True` avoids squaring same features (e.g., `Age^2`)
- Useful for capturing nonlinear interactions

---

## 🧱 Section 5: Prepare Input for Modeling

```python
X = train.drop(columns=['id', 'Calories'])
y = np.log1p(train['Calories'])  
X_test = test.drop(columns=['id'])
```

### 📝 Notes:
- Drop irrelevant columns like `id`
- Target is log-transformed (`np.log1p`) to stabilize large values
- RMSLE metric works well with log targets

---

## 🤖 Section 6: Define Models

```python
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

models = {
    'CatBoost': CatBoostRegressor(...),
    'XGBoost': XGBRegressor(...),
    'LightGBM': LGBMRegressor(...)
}
```

### 📝 Notes:
| Model | Strengths | Notes |
|------|-----------|--------|
| CatBoost | Handles categorical variables natively | Great for tabular data |
| XGBoost | Accurate but slower | Good for final submissions |
| LightGBM | Fastest training | Best for quick iterations |

---

## 🏋️‍♂️ Section 7: Cross-Validation Training Loop

```python
from sklearn.model_selection import KFold
kf = KFold(n_splits=7, shuffle=True, random_state=42)

results = {name: {'oof': ..., 'pred': ..., 'rmsle': ...} for name in models}

for name, model in models.items():
    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y)):
        # Train and evaluate
```

### 📝 Notes:
- 7-Fold Cross-Validation ensures robustness
- Stores Out-of-Fold (OOF) predictions for evaluation
- Predicts on test set and averages across folds

---

## 🧮 Section 8: Blend Predictions Using Optimization

```python
from scipy.optimize import minimize

def rmsle_loss(weights):
    blended = weights[0] * oof_preds['CatBoost'] + ...
    return np.sqrt(mean_squared_log_error(y_true, blended))

res = minimize(rmsle_loss, initial_weights, method='SLSQP', ...)
best_weights = res.x
```

### 📝 Notes:
- Blending finds optimal weightings between model predictions
- `minimize` from `scipy.optimize` finds best blend
- Prevents manual guessing of weights

---

## 📦 Section 9: Final Submission

```python
blended_preds = (
    best_weights[0] * test_preds['CatBoost'] +
    best_weights[1] * test_preds['XGBoost'] +
    best_weights[2] * test_preds['LightGBM']
)
blended_preds = np.clip(blended_preds, 1, 314)
submission['Calories'] = blended_preds
submission.to_csv('submission.csv', index=False)
```

### 📝 Notes:
- Clip predictions between 1 and 314 per competition rules
- Save submission file with correct column name

---

## 🧩 Section 10: Ensemble with Other Submissions (Optional)

```python
df1 = pd.read_csv("submission1.csv")
df2 = pd.read_csv("submission2.csv")
df3 = pd.read_csv("submission3.csv")

ground_truth['Calories'] = 0.4 * df1['Calories'] + 0.3 * df2['Calories'] + 0.3 * df3['Calories']
```

### 📝 Notes:
- Ensembling improves score by combining diverse models
- Weighted average based on performance or optimization
- You can also use a meta-model to combine them

---

## 📈 Future Improvements Stack

| Area | Suggestion |
|------|------------|
| Feature Engineering | Try PCA, AutoML feature generation |
| Hyperparameter Tuning | Use Optuna or BayesianOptimization |
| Stacking | Add a meta-model (like LinearRegression or MLP) |
| GPU Usage | Enable tree_method='gpu_hist' in XGBoost |
| Post-processing | Use quantile regression or outlier smoothing |
| Model Diversity | Add Random Forest, Extra Trees, or TabNet |

---

## 📝 Student Assignment (Grading Format)

### 🧪 Task:
Reproduce this notebook on your own dataset (can be synthetic or from Kaggle Playground Series).

### 📋 Requirements:

| Component | Points |
|----------|--------|
| Load and explore data correctly | 10 |
| Implement all feature engineering functions | 20 |
| Encode categorical features properly | 10 |
| Train at least two models with cross-validation | 20 |
| Evaluate using RMSLE | 10 |
| Blend predictions using optimization | 15 |
| Submit final CSV with correct formatting | 10 |
| Bonus: Add stacking or ensemble with other submissions | +5 |

---

## 📚 Summary of Techniques Learned

| Technique | Description |
|----------|-------------|
| Feature Engineering | Create new features to help models learn better |
| Label Encoding | Convert categorical features to numbers |
| Polynomial Features | Generate higher-order feature interactions |
| Cross-Validation | Evaluate model performance robustly |
| Gradient Boosting Models | Powerful tree-based algorithms for tabular data |
| RMSLE Metric | Measures error on log scale |
| Ensemble Blending | Combine multiple models for better accuracy |
| Optimization | Find best weights automatically |

---





---

## 🧠 Interview Questions & Answers

---

### 🔹 Q1: What feature engineering techniques did you apply in this notebook?

**Answer:**
The following feature engineering techniques were applied:

1. **Cross Term Features** (`add_feature_cross_terms`)
   - Multiplies all unique pairs of numerical features (e.g., `Age × Weight`)
   - Captures interactions between variables
2. **Interaction Features** (`add_interaction_features`)
   - Adds/subtracts/divides feature pairs
   - Example: `Age + Duration`, `Heart_Rate / Body_Temp`
3. **Statistical Features** (`add_statistical_features`)
   - Computes row-wise mean, standard deviation, max, min, and median
   - Helps models understand overall behavior of each sample
4. **Polynomial Features** (`PolynomialFeatures`)
   - Generates degree-2 interaction-only combinations
   - Enhances model’s ability to capture nonlinear patterns

**Code Snippet:**
```python
train = add_feature_cross_terms(train, numerical_features)
train = add_interaction_features(train, numerical_features)
train = add_statistical_features(train, numerical_features)
```

---

### 🔹 Q2: Why is label encoding used, and how is it applied in the code?

**Answer:**
Label Encoding converts categorical variables into numerical form so that machine learning algorithms can process them.

In this notebook:
- The `'Sex'` column (`Male/Female`) was encoded as `0/1`
- Used `LabelEncoder()` from `sklearn.preprocessing`

**Code Snippet:**
```python
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train['Sex'] = le.fit_transform(train['Sex'])
test['Sex'] = le.transform(test['Sex'])
```

> ✅ Note: `.fit_transform()` is used only on the train set to avoid data leakage.

---

### 🔹 Q3: How are polynomial features used here, and why are they useful?

**Answer:**
Polynomial features generate higher-order combinations of input features to help models learn complex relationships.

- Degree=2, interaction_only=True → generates only cross terms like `Age × Duration`
- No squared terms (like `Age²`)
- Applied using `PolynomialFeatures`

**Code Snippet:**
```python
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_train = poly.fit_transform(train[numerical_features])
poly_test = poly.transform(test[numerical_features])
```

This helps gradient boosting models better capture non-linearities in the data.

---

### 🔹 Q4: Why is the target variable log-transformed?

**Answer:**
The target variable `'Calories'` is log-transformed using `np.log1p()` to stabilize variance and reduce skewness.

- `log1p(x)` = log(1 + x), handles zero values safely
- Improves performance when using RMSLE metric

**Code Snippet:**
```python
y = np.log1p(train['Calories'])  # Log transform target
```

> ⚠️ When making predictions, use `np.expm1()` to revert back to original scale.

---

### 🔹 Q5: Explain what KFold Cross-Validation does and why it's used here.

**Answer:**
KFold Cross-Validation splits the dataset into `k` equal parts (folds), trains on `k-1` folds, and validates on 1 fold — repeating `k` times.

- Ensures robust evaluation by reducing variance due to random sampling
- Prevents overfitting and gives more reliable performance estimate

**Code Snippet:**
```python
from sklearn.model_selection import KFold
kf = KFold(n_splits=7, shuffle=True, random_state=42)
```

Used with CatBoost, XGBoost, and LightGBM to evaluate RMSLE across 7 folds.

---

### 🔹 Q6: Which models are trained, and why were they chosen?

**Answer:**
Three gradient boosting models were used:

| Model | Description |
|-------|-------------|
| **CatBoost** | Handles categorical features natively; fast and accurate |
| **XGBoost** | Powerful but slower; supports early stopping |
| **LightGBM** | Fastest training speed; memory efficient |

Each has different strengths, so ensembling improves final performance.

**Code Snippet:**
```python
models = {
    'CatBoost': CatBoostRegressor(...),
    'XGBoost': XGBRegressor(...),
    'LightGBM': LGBMRegressor(...)
}
```

---

### 🔹 Q7: What is Out-of-Fold (OOF) prediction, and how is it used?

**Answer:**
Out-of-Fold (OOF) prediction means storing predictions made during cross-validation on the validation part of each fold.

- Used to evaluate final ensemble
- Avoids overfitting by ensuring predictions are made on unseen data

**Code Snippet:**
```python
results[name]['oof'][valid_idx] = oof_pred
```

These OOF predictions are later used to find optimal blending weights.

---

### 🔹 Q8: How is the final submission created using model blending?

**Answer:**
Model predictions are blended using **optimization** to find the best weightings that minimize RMSLE on OOF predictions.

- Uses `scipy.optimize.minimize()` with constraints
- Final blend uses optimized weights

**Code Snippet:**
```python
def rmsle_loss(weights):
    blended = (
        weights[0] * oof_preds['CatBoost'] +
        weights[1] * oof_preds['XGBoost'] +
        weights[2] * oof_preds['LightGBM']
    )
    return np.sqrt(mean_squared_log_error(y_true, blended))
```

Final test predictions are combined using these weights and clipped to `[1, 314]`.

---

### 🔹 Q9: Why do we clip predictions before submission?

**Answer:**
Some competitions restrict predicted values to a specific range (e.g., 1 to 314 in this case). Clipping ensures predictions stay within bounds and avoids disqualification.

**Code Snippet:**
```python
blended_preds = np.clip(blended_preds, 1, 314)
```

---

### 🔹 Q10: What is stacking, and how is it implemented in this notebook?

**Answer:**
Stacking combines predictions from multiple models using a meta-model (not shown directly in this notebook).

However, an **optimized weighted average** is used instead, which is a simple form of stacking.

- Finds optimal weights via optimization
- Can be extended to use Linear Regression or Neural Network as meta-learner

**Code Snippet:**
```python
blended_preds = (
    best_weights[0] * test_preds['CatBoost'] +
    best_weights[1] * test_preds['XGBoost'] +
    best_weights[2] * test_preds['LightGBM']
)
```

---

## 📌 Summary Table of Key Concepts

| Concept | Purpose | Code Reference |
|--------|---------|----------------|
| Feature Engineering | Improve model performance | `add_*` functions |
| Label Encoding | Handle categorical variables | `LabelEncoder()` |
| Polynomial Features | Capture non-linear interactions | `PolynomialFeatures()` |
| Log Transform | Stabilize skewed target | `np.log1p()` |
| KFold CV | Robust model evaluation | `KFold()` |
| Gradient Boosting | Powerful tabular data models | `CatBoost`, `XGBoost`, `LGBM` |
| OOF Predictions | Evaluate ensemble fairly | `results['oof']` |
| Blending | Combine model predictions | `minimize()` |
| Prediction Clipping | Stay within competition rules | `np.clip()` |

---





### 📊 1. Load Data

```
┌──────────────┐
│   Load CSVs  │
│ train.csv    │
│ test.csv     │
│ submission.csv│
└──────────────┘
```

---

### 🧬 2. Feature Engineering

```
┌────────────────────────────┐
│ Add Feature Cross Terms    │
│ (e.g., Age × Weight)       │
├────────────────────────────┤
│ Add Interaction Features   │
│ (+, −, ÷ between features) │
├────────────────────────────┤
│ Add Row-wise Statistics    │
│ mean, std, max, min, median│
└────────────────────────────┘
```

---

### 🔤 3. Encode Categorical Variables

```
┌─────────────────────────┐
│ Label Encode 'Sex'      │
│ Male → 0, Female → 1    │
│ Convert to category type│
└─────────────────────────┘
```

---

### 📐 4. Polynomial Feature Expansion

```
┌──────────────────────────────────────────────┐
│ Generate degree-2 interaction-only features  │
│ e.g., Age × Duration                         │
└──────────────────────────────────────────────┘
```

---

### 📦 5. Prepare Input for Modeling

```
┌──────────────────────────────────────────────┐
│ Drop ID and target column                   │
│ X = train.drop(['id', 'Calories'])          │
│ y = log1p(Calories)                          │
│ X_test = test.drop('id')                    │
└──────────────────────────────────────────────┘
```

---

### 🤖 6. Train Models with Cross-Validation

```
┌──────────────────────────────────────────────┐
│ Define Models: CatBoost, XGBoost, LightGBM  │
├──────────────────────────────────────────────┤
│ Use 7-Fold KFold CV                          │
├──────────────────────────────────────────────┤
│ Train each model on folds                    │
│ Store OOF predictions                        │
│ Evaluate RMSLE per fold                      │
└──────────────────────────────────────────────┘
```

---

### 🧮 7. Blend Predictions Using Optimization

```
┌──────────────────────────────────────────────┐
│ Revert log predictions using expm1()         │
├──────────────────────────────────────────────┤
│ Define RMSLE loss function                   │
├──────────────────────────────────────────────┤
│ Optimize weights using scipy.minimize        │
├──────────────────────────────────────────────┤
│ Apply best weights to test predictions       │
└──────────────────────────────────────────────┘
```

---

### 📤 8. Final Submission

```
┌──────────────────────────────────────────────┐
│ Clip predictions between 1 and 314           │
├──────────────────────────────────────────────┤
│ Save final submission CSV                    │
└──────────────────────────────────────────────┘
```

---

### 🧩 9. Ensemble with Other Submissions (Optional)

```
┌──────────────────────────────────────────────┐
│ Load multiple submissions from other models  │
├──────────────────────────────────────────────┤
│ Blend using weighted average                 │
│ e.g., 0.4 * df1 + 0.3 * df2 + 0.3 * df3      │
├──────────────────────────────────────────────┤
│ Save final ensemble submission               │
└──────────────────────────────────────────────┘
```

---

## 🖼️ Full Pipeline Diagram (Text Version)

```
[Load Data]
    ↓
[Feature Engineering]
    ↓
[Encode Categoricals]
    ↓
[Polynomial Features]
    ↓
[Prepare Input for Model]
    ↓
[Train Models with Cross-Validation]
    ↓
[Blend Predictions with Optimization]
    ↓
[Create Final Submission]
    ↓
[Ensemble (Optional)]
```

---


